# Pipeline reproducible de análisis estadístico

## Actividad antifúngica de extractos de tomillo (*Thymus vulgaris*) frente a *Fusarium* spp.

Este notebook es el **orquestador educativo** de una pipeline estadística
automatizada, reproducible y reutilizable. Todo el código de cómputo pesado
vive en el paquete `pipeline/` (importable); aquí se explica cada fase, se
ejecuta y se interpretan los resultados.

### Contexto del proyecto

Se evaluó la actividad antifúngica de extractos de tomillo obtenidos por tres
técnicas de extracción (**maceración**, **Soxhlet** y **ultrasonido**) frente
a **31 aislados de *Fusarium* spp.**, todos ensayados a una única concentración
de **5 mg/mL**. Para cada combinación técnica × aislado se dispone de **3
réplicas biológicas** (unidad experimental: caja Petri), totalizando 279
observaciones de bioensayo, más 9 mediciones de rendimiento de extracción.

### Objetivos científicos

1. **Rendimiento de extracción**: determinar si la técnica afecta el rendimiento (%).
2. **Inhibición del crecimiento micelial**: comparar la actividad antifúngica
   de las técnicas y de los aislados.
3. **Inhibición de la producción de conidias**: evaluar el efecto sobre la
   esporulación (variable continua en escala log10).
4. **Susceptibilidad relativa de los aislados**: agrupar los aislados según su
   perfil de susceptibilidad (NUNCA se usa el término "resistente" sin un
   criterio validado).
5. **Ranking de técnicas**: integrar rendimiento y actividad en un score.
6. **Reproducibilidad**: que todo el análisis sea re-ejecutable con datos nuevos.

### Estructura de las 12 fases

| Fase | Módulo | Contenido |
|------|--------|-----------|
| 1-2 | `cargar_datos` | Carga de datos y auditoría de calidad |
| 3 | `limpiar` | Normalización del dataset maestro |
| 4 | `eda` | Exploración y descriptivos |
| 5 | `diseno` | Inferencia del diseño experimental |
| 6 | `supuestos` | Verificación de supuestos de los modelos |
| 7 | `modelos` | Análisis inferencial y selección automática |
| 8 | `comparaciones` | Comparaciones múltiples y letras CLD |
| 9 | `visualizar` | Figuras de resultados |
| 10 | `multivariado` | PCA, clustering y categorías biológicas |
| 11 | `ranking` | Ranking de técnicas |
| 12 | `informe` | Informe final (Markdown y HTML) |

> **Contrato científico**: se respetan las reglas del proyecto: no se eliminan
> atípicos automáticamente, no se llama "resistente" a ningún aislado, se
> reportan efecto e IC además del p-valor, y se distingue significancia
> estadística de significancia biológica.


## Configuración del entorno

Esta celda prepara el entorno: determina la raíz del proyecto, agrega el
paquete `pipeline/` al `sys.path`, fija la semilla aleatoria global (42) para
garantizar la reproducibilidad de los procedimientos estocásticos (KMeans,
PCA, etc.) y verifica las versiones de las librerías utilizadas.

**Por qué una semilla fija**: el análisis de clusters y cualquier procedimiento
con inicialización aleatoria producen resultados ligeramente distintos entre
ejecuciones. Fijar `random_state`/`seed` hace que el análisis sea determinista.


In [1]:

import os
import sys
import warnings
from pathlib import Path

# Determinar la raíz del proyecto (directorio que contiene pipeline/)
RAIZ = Path(os.getcwd()).resolve()
for candidato in (RAIZ, RAIZ.parent, RAIZ.parent.parent):
    if (candidato / "pipeline" / "config.py").exists():
        RAIZ = candidato
        break
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
os.chdir(RAIZ)
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

from pipeline.config import (
    fijar_semilla, METODOS, METODO_LABEL, VARIABLES_RESPUESTA, VARIABLE_LABEL,
)
from pipeline import (
    cargar_datos, limpiar, eda, diseno, supuestos, modelos, comparaciones,
    visualizar, multivariado, ranking, informe,
)

fijar_semilla(42)

print("Versión de librerías:")
import importlib
for nombre in ("pandas", "numpy", "scipy", "statsmodels", "sklearn", "pingouin", "matplotlib", "seaborn"):
    try:
        mod = importlib.import_module(nombre)
        print(f"  {nombre} {mod.__version__}")
    except Exception as exc:  # pragma: no cover
        print(f"  {nombre}: {exc}")

# Diccionario acumulador de resultados para el informe final
resultados = {}
print("\nRaíz del proyecto:", RAIZ)


Versión de librerías:
  pandas 3.0.3
  numpy 2.5.0
  scipy 1.18.0
  statsmodels 0.14.6
  sklearn 1.9.0


  pingouin 0.6.1
  matplotlib 3.11.0
  seaborn 0.13.2

Raíz del proyecto: /home/mniev/projects/proyecto_tomillo


## Fase 1: Carga de datos

**Qué se hace**: se leen las dos fuentes de datos de la pipeline:
- `dca/resultados/database/consolidado_tidy.xlsx` (hoja "Consolidado", 279 filas):
  bioensayo con crecimiento micelial (mm), % de inhibición micelial, conidias
  (log10/mL) e inhibición de conidias para cada técnica × aislado × réplica.
- `dca/resultados/database/rendimiento_extraccion.csv` (9 filas): rendimiento de
  extracción por técnica y réplica biológica.

**Por qué**: se usa el dataset consolidado y limpio (no el Excel crudo) como
fuente única de entrada; esto garantiza trazabilidad.

**Cómo interpretar**: se verifica que las dimensiones coincidan con lo esperado
(279 y 9 filas) y se inspeccionan columnas, tipos y primeras filas antes de
cualquier transformación.


In [2]:

datos = cargar_datos.cargar_datos()
resultados["datos"] = datos


FASE 1 - Carga de datos

[Bioensayo]
  Dimensiones: 279 filas x 7 columnas
  Columnas:    ['Metodo de extraccion', 'Aislamiento', 'Replica', 'Crecimiento micelial (mm)', '%INH micelial', 'Conidias (log10/ml)', '%INH conidias']
  Tipos:       {'Metodo de extraccion': 'str', 'Aislamiento': 'str', 'Replica': 'str', 'Crecimiento micelial (mm)': 'int64', '%INH micelial': 'float64', 'Conidias (log10/ml)': 'float64', '%INH conidias': 'float64'}
  Vista previa (5 filas):
  Metodo de extraccion Aislamiento Replica  Crecimiento micelial (mm)  %INH micelial  Conidias (log10/ml)  %INH conidias
0           Maceración         HC3      R1                          0     100.000000                 0.00     100.000000
1           Maceración         HC3      R2                          0     100.000000                 0.00     100.000000
2           Maceración         HC3      R3                          0     100.000000                 4.85      37.012987
3           Maceración         HC5      R1      

## Fase 2: Auditoría de calidad de datos

**Qué se hace**: se audita el dataset columna a columna: valores faltantes,
filas duplicadas exactas, columnas constantes, tipos, valores inconsistentes
(método en {Maceración, Soxhlet, Ultrasonido}, réplica en {R1,R2,R3}, %INH
micelial en [0,100], crecimiento en [0,100], conidias >= 0) y atípicos por
rango intercuartílico (1.5xIQR).

**Por qué**: la auditoría previa es obligatoria para detectar duplicados,
faltantes y valores imposibles antes del análisis (regla de integridad del
proyecto).

**Supuestos**: se asume que los límites de cada variable son los documentados
en el diccionario de datos; el %INH de conidias puede ser negativo por
diseño (el extracto puede inducir mayor esporulación) y por eso no se
restringe su rango.

**Cómo interpretar**: los atípicos se **flaguean pero nunca se eliminan**;
su existencia se documenta y su influencia se evalúa en los análisis.
Valores "inconsistentes" > 0 indicarían errores de registro que requieren
revisión con el investigador.


In [3]:

tabla_auditoria, resumen_auditoria = cargar_datos.auditoria_calidad(datos["bio"])
resultados["auditoria"] = {"tabla": tabla_auditoria, "resumen": resumen_auditoria}
display(tabla_auditoria)


FASE 2 - Auditoría de calidad de datos
Filas: 279 | Columnas: 7 | Filas duplicadas exactas: 0
                 variable tipo_dato  n_no_nulos  pct_faltantes  columna_constante  n_filas_duplicadas  n_valores_inconsistentes  n_atipicos_iqr
     Metodo de extraccion       str         279            0.0              False                   0                         0               0
              Aislamiento       str         279            0.0              False                   0                         0               0
                  Replica       str         279            0.0              False                   0                         0               0
Crecimiento micelial (mm)     int64         279            0.0              False                   0                         0               0
            %INH micelial   float64         279            0.0              False                   0                         0               1
      Conidias (log10/ml)   float64       

,variable,tipo_dato,n_no_nulos,pct_faltantes,columna_constante,n_filas_duplicadas,n_valores_inconsistentes,n_atipicos_iqr
0,Metodo de extraccion,str,279,0.0,False,0,0,0
1,Aislamiento,str,279,0.0,False,0,0,0
2,Replica,str,279,0.0,False,0,0,0
3,Crecimiento micelial (mm),int64,279,0.0,False,0,0,0
4,%INH micelial,float64,279,0.0,False,0,0,1
5,Conidias (log10/ml),float64,279,0.0,False,0,0,8
6,%INH conidias,float64,279,0.0,False,0,0,9


## Fase 3: Limpieza y normalización del dataset maestro

**Qué se hace**: se renombran las columnas al esquema canónico en inglés
(`metodo_extraccion`, `aislamiento`, `replica`, `crecimiento_micelial_mm`,
`porcentaje_inhibicion_micelial`, `conidias_log10_ml`,
`porcentaje_inhibicion_conidias`), se normaliza el método a minúsculas sin
acentos y la réplica a entero 1-3. Se **valida** que el dataset cumpla: 279
filas, sin valores nulos y diseño balanceado (31 aislados × 3 métodos × 3
réplicas). Luego se incorporan los **controles C4** del Excel crudo del
laboratorio (columnas `control_crecimiento_mm` y `control_conidias_log10`,
una por aislado y compartidas por sus 3 réplicas), dejando el **dataset
maestro** con 9 columnas. Se guarda en CSV y XLSX (hojas Bioensayo y
Rendimiento) junto con el diccionario de datos.

**Por qué**: un esquema canónico estable facilita reutilizar la pipeline con
nuevos archivos y evita errores de codificación (acentos, mayúsculas); los
controles C4 explícitos permiten validar la fórmula del %INH (integridad de
datos) y sirven de línea de base.

**Supuestos**: el dataset consolidado ya contiene los %INH calculados contra el
control C4 de cada aislado; la concentración es constante (5 mg/mL) y no se
modela. Si el Excel crudo no está disponible, los controles no se incorporan
y la validación del %INH se saltea con un aviso (sin romper el flujo).

**Cómo interpretar**: si la validación lanzara un error, habría que resolver el
problema de datos antes de continuar; un diseño desbalanceado cambiaría la
selección de modelos.


In [4]:

df_bio, df_rend = limpiar.normalizar_master(datos["bio"], datos["rend"])
limpiar.guardar_master(df_bio, df_rend)
resultados["master"] = {"bio": df_bio, "rend": df_rend}
display(df_bio.head(8))


  Controles C4 incorporados: 93 filas, sin nulos.


FASE 3 - Dataset maestro normalizado y validado
  Bioensayo: 279 filas x 9 columnas
  Aislados unicos: 31
  Celdas método × aislado con 3 réplicas: True
  Guardado: /home/mniev/projects/proyecto_tomillo/dca/resultados/database/master_dataset_tomillo_fusarium.csv
  Guardado: /home/mniev/projects/proyecto_tomillo/dca/resultados/database/master_dataset_tomillo_fusarium.xlsx
  Diccionario: /home/mniev/projects/proyecto_tomillo/dca/resultados/tablas/diccionario_datos.md

Vista previa del dataset maestro:
  metodo_extraccion aislamiento  replica  crecimiento_micelial_mm  porcentaje_inhibicion_micelial  conidias_log10_ml  porcentaje_inhibicion_conidias  control_crecimiento_mm  control_conidias_log10
0        maceracion         HC3        1                        0                      100.000000               0.00                      100.000000                    68.0                     7.7
1        maceracion         HC3        2                        0                      100.000000    

,metodo_extraccion,aislamiento,replica,crecimiento_micelial_mm,porcentaje_inhibicion_micelial,conidias_log10_ml,porcentaje_inhibicion_conidias,control_crecimiento_mm,control_conidias_log10
0,maceracion,HC3,1,0,100.000000,0.00,100.000000,68.0,7.70
1,maceracion,HC3,2,0,100.000000,0.00,100.000000,68.0,7.70
2,maceracion,HC3,3,0,100.000000,4.85,37.012987,68.0,7.70
3,maceracion,HC5,1,18,60.869565,5.53,16.212121,46.0,6.60
4,maceracion,HC5,2,12,73.913043,5.69,13.787879,46.0,6.60
5,maceracion,HC5,3,18,60.869565,5.32,19.393939,46.0,6.60
6,maceracion,HC6,1,16,70.370370,7.03,5.383580,54.0,7.43
7,maceracion,HC6,2,12,77.777778,6.08,18.169583,54.0,7.43


## Fase 3.5: Validación de la fórmula del %INH (integridad de datos)

**Qué se hace**: se reconstruye el %INH con la fórmula
`%INH = (1 - C1/C4) × 100` usando las columnas de control C4 incorporadas en la
Fase 3 y se compara contra los valores reportados por el laboratorio
(`porcentaje_inhibicion_micelial` y `porcentaje_inhibicion_conidias`), fila por
fila. Se reporta la máxima diferencia absoluta, el número de discrepancias y el
estado.

**Por qué**: es una verificación de integridad de datos: confirma que los %INH
reportados son consistentes con la fórmula documentada y con los controles.
**No reemplaza** las respuestas reportadas: la inferencia principal usa el
%INH reportado por el investigador (instrucción explícita).

**Supuestos**: para las conidias, la fórmula se aplica sobre la escala log10
directamente, porque el laboratorio reportó la reducción de conidias en log10
(no en conteos crudos). Si un control fuera 0, el %INH verificado se define
como 100 cuando C1=0 y como indefinido (NaN) cuando C1>0; en los datos reales
el control nunca es 0.

**Cómo interpretar**: si el estado es 'ok' (sin discrepancias por encima de la
tolerancia 1e-6), la fórmula del laboratorio es consistente con los datos; si
hubiera discrepancias, habría que investigar si se deben a escala (log10 vs
crudo) o a redondeo antes de reportar.


In [5]:

validacion_inh = limpiar.validar_inh(df_bio)
resultados["validacion_inh"] = validacion_inh


FASE 7.5 - Validación de la fórmula del %INH (informativa)
  La inferencia principal usa el %INH reportado por el laboratorio;
  esta validación solo verifica la integridad de los datos.
                      variable  n_verificadas  max_diff_abs  n_discrepancias estado                                                                           nota
porcentaje_inhibicion_micelial            279           0.0                0     ok La fórmula (1 - C1/C4) x 100 coincide con el %INH reportado (tolerancia 1e-6).
porcentaje_inhibicion_conidias            279           0.0                0     ok La fórmula (1 - C1/C4) x 100 coincide con el %INH reportado (tolerancia 1e-6).


## Fase 4: Análisis exploratorio (EDA)

**Qué se hace**: se calculan descriptivos por método (n, media, DE, error
estándar, IC95%, mínimo y máximo) para las cuatro variables de respuesta, y se
generan figuras exploratorias: histogramas, boxplots, violin plots, densidades
con rug, QQ-plots, matriz de correlación (Pearson con p) y un scatter entre
crecimiento y conidias.

**Por qué**: el EDA revela la forma de las distribuciones (asimetrías, efecto
techo en %INH micelial), la variabilidad entre métodos y las relaciones entre
respuestas, orientando la elección del modelo.

**Supuestos**: ninguno inferencial; es una fase descriptiva.

**Cómo interpretar**:
- Si el %INH micelial se concentra en 100 (efecto techo), la distribución es
  asimétrica y los métodos no paramétricos o modelos mixtos ganan relevancia.
- Si el %INH de conidias toma valores negativos, el extracto indujo mayor
  esporulación en esas celdas (la escala es log10).
- Correlaciones altas entre respuestas sugieren que la inhibición micelial y
  la de conidias covarían (a revisar en el multivariado).


In [6]:

tabla_desc = eda.resumen_descriptivo(df_bio)
resultados["eda"] = {"descriptivos": tabla_desc}
display(tabla_desc)

figuras_eda = eda.figuras_eda(df_bio)
resultados["eda"]["figuras"] = figuras_eda


FASE 4 - Resumen descriptivo por método
metodo_extraccion                              variable  n  media  desviacion_estandar  error_estandar  ic95_inferior  ic95_superior  minimo  maximo
       Maceración             Crecimiento micelial (mm) 93  7.484                8.546           0.886          5.724          9.244   0.000  28.000
       Maceración               Inhibición micelial (%) 93 86.349               15.913           1.650         83.072         89.626  44.000 100.000
       Maceración                   Conidias (log10/mL) 93  5.268                1.780           0.185          4.902          5.635   0.000   7.560
       Maceración            Inhibición de conidias (%) 93 29.239               23.748           2.463         24.349         34.130  -1.449 100.000
       Maceración Control C4: crecimiento micelial (mm) 93 56.677               11.009           1.142         54.410         58.945  26.000  75.000
       Maceración       Control C4: conidias (log10/mL) 93  7.392 

,metodo_extraccion,variable,n,media,desviacion_estandar,error_estandar,ic95_inferior,ic95_superior,minimo,maximo
0,Maceración,Crecimiento micelial (mm),93,7.484,8.546,0.886,5.724,9.244,0.000,28.000
1,Maceración,Inhibición micelial (%),93,86.349,15.913,1.650,83.072,89.626,44.000,100.000
2,Maceración,Conidias (log10/mL),93,5.268,1.780,0.185,4.902,5.635,0.000,7.560
3,Maceración,Inhibición de conidias (%),93,29.239,23.748,2.463,24.349,34.130,-1.449,100.000
4,Maceración,Control C4: crecimiento micelial (mm),93,56.677,11.009,1.142,54.410,58.945,26.000,75.000
5,Maceración,Control C4: conidias (log10/mL),93,7.392,0.481,0.050,7.293,7.491,6.180,8.030
6,Soxhlet,Crecimiento micelial (mm),93,22.204,6.877,0.713,20.788,23.621,10.000,40.000
7,Soxhlet,Inhibición micelial (%),93,59.593,13.617,1.412,56.788,62.397,13.043,84.375
8,Soxhlet,Conidias (log10/mL),93,7.041,0.566,0.059,6.924,7.157,5.780,8.260
9,Soxhlet,Inhibición de conidias (%),93,4.432,9.194,0.953,2.539,6.326,-32.584,24.067


  Figuras EDA generadas: 16


## Fase 5: Diseño experimental

**Qué se hace**: se infiere y documenta el diseño: DCA factorial
técnica × aislado, 3 réplicas biológicas, unidad experimental (caja Petri),
concentración única (5 mg/mL) y balanceo.

**Por qué**: conocer el diseño determina la estructura del modelo (factores
fijos/aleatorios, interacciones) y las limitaciones de la inferencia.

**Supuestos y caveats**:
- El aislado se trata como factor **fijo** en el ANOVA factorial y como
  **aleatorio** en el análisis de sensibilidad LMM.
- Cada %INH se calculó contra **un único control C4 compartido** por las tres
  réplicas del aislado: las réplicas de %INH no son totalmente independientes
  (pseudorreplicación del control). El crecimiento crudo en mm no presenta
  este problema.

**Cómo interpretar**: la tabla resume el diseño inferido; el texto resalta las
limitaciones que condicionarán las conclusiones (especialmente para %INH).


In [7]:

diseno_info = diseno.inferir_diseno(df_bio)
resultados["diseno"] = diseno_info
display(diseno_info["detalle"])


FASE 5 - Diseño experimental inferido
                                 atributo                                                                                               valor
                           Tipo de diseño                                                                      DCA factorial método × aislado
                                 Factores                              método de extracción (fijo); aislado (fijo en ANOVA, aleatorio en LMM)
                        Niveles de método                                                                                                   3
                       Niveles de aislado                                                                                                  31
Número de tratamientos (método × aislado)                                                                                                  93
            Réplicas biológicas por celda                                                                     

,atributo,valor
0,Tipo de diseño,DCA factorial método × aislado
1,Factores,método de extracción (fijo); aislado (fijo en ...
2,Niveles de método,3
3,Niveles de aislado,31
4,Número de tratamientos (método × aislado),93
5,Réplicas biológicas por celda,3
6,Unidad experimental,Caja Petri
7,Concentración ensayada (mg/mL),5.0
8,Variables de respuesta,Crecimiento micelial (mm); Inhibición micelial...
9,Diseño balanceado,Sí


## Fase 6: Verificación de supuestos

**Qué se hace**: para cada variable de respuesta se ajusta el modelo OLS
factorial `variable ~ método * aislado` y se evalúa:
- **Normalidad** de residuos (Shapiro-Wilk),
- **Homocedasticidad** entre métodos (Levene y Bartlett),
- **Independencia** (estadístico de Durbin-Watson),
y se generan figuras de diagnóstico (histograma, QQ-plot, residuos vs
ajustados, residuos por método).

**Por qué**: el ANOVA factorial exige residuos normales, homocedásticos e
independientes; si no se cumplen, la inferencia debe apoyarse en la vía no
paramétrica (fase 7) o en modelos mixtos.

**Supuestos**: p > 0.05 se interpreta como ausencia de evidencia en contra del
supuesto (no como prueba de su cumplimiento).

**Cómo interpretar**:
- Shapiro p < 0.05 con %INH micelial refleja el efecto techo (muchos valores
  en 100).
- Bartlett puede no ser calculable si un grupo es constante (varianza nula),
  por ejemplo en crecimiento 0 (inhibición completa); se reporta como tal.
- Durbin-Watson cercano a 2 sugiere independencia de los residuos.


In [8]:

supuestos_result = {}
for variable in VARIABLES_RESPUESTA:
    supuestos_result[variable] = supuestos.verificar_supuestos(df_bio, variable)
    display(supuestos_result[variable]["tabla_supuestos"])
resultados["supuestos"] = supuestos_result


[Crecimiento micelial (mm)] Shapiro-Wilk p=0.0328; Levene p=0.0628; Bartlett p=0.0819; Durbin-Watson=2.71


,test,estadistico,p_valor,interpretacion
0,Shapiro-Wilk (residuos),0.9890,0.0328,Se rechaza normalidad de los residuos (p<0.05).
1,Levene (por metodo),2.7964,0.0628,Homocedasticidad aceptada (p>0.05).
2,Bartlett (por metodo),5.0053,0.0819,Homocedasticidad aceptada (p>0.05).
3,Durbin-Watson (independencia),2.7061,NaN,Posible autocorrelacion (DW fuera de 1.5-2.5).


[Inhibición micelial (%)] Shapiro-Wilk p=0.0788; Levene p=0.0982; Bartlett p=0.0760; Durbin-Watson=2.70


,test,estadistico,p_valor,interpretacion
0,Shapiro-Wilk (residuos),0.9908,0.0788,No se rechaza normalidad de los residuos (p>0....
1,Levene (por metodo),2.3399,0.0982,Homocedasticidad aceptada (p>0.05).
2,Bartlett (por metodo),5.1538,0.0760,Homocedasticidad aceptada (p>0.05).
3,Durbin-Watson (independencia),2.7034,NaN,Posible autocorrelacion (DW fuera de 1.5-2.5).


[Conidias (log10/mL)] Shapiro-Wilk p=0.0000; Levene p=0.0000; Bartlett p=0.0000; Durbin-Watson=2.46


,test,estadistico,p_valor,interpretacion
0,Shapiro-Wilk (residuos),0.8021,0.0,Se rechaza normalidad de los residuos (p<0.05).
1,Levene (por metodo),13.5316,0.0,Heterocedasticidad detectada (p<0.05).
2,Bartlett (por metodo),186.2385,0.0,Heterocedasticidad detectada (p<0.05).
3,Durbin-Watson (independencia),2.4556,NaN,Independencia razonable (DW cercano a 2).


[Inhibición de conidias (%)] Shapiro-Wilk p=0.0000; Levene p=0.0001; Bartlett p=0.0000; Durbin-Watson=2.41


,test,estadistico,p_valor,interpretacion
0,Shapiro-Wilk (residuos),0.7731,0.0000,Se rechaza normalidad de los residuos (p<0.05).
1,Levene (por metodo),9.5376,0.0001,Heterocedasticidad detectada (p<0.05).
2,Bartlett (por metodo),124.8739,0.0000,Heterocedasticidad detectada (p<0.05).
3,Durbin-Watson (independencia),2.4128,NaN,Independencia razonable (DW cercano a 2).


## Fase 7: Análisis inferencial y selección automática de modelos

**Qué se hace**:
1. **Rendimiento**: ANOVA de una vía (OLS) `rendimiento_pct ~ método` con
   tabla tipo II, eta², omega² y supuestos; si fallan, se complementa con
   Kruskal-Wallis.
2. **Factorial por variable**: se ajusta `variable ~ método * aislado` con
   ANOVA tipo II y tamaños de efecto (eta² y omega² parciales). La selección
   de la vía de inferencia es **automática y justificada**:
   - Si la variable es un **conteo entero >= 0 sobredisperso** -> rama GLM
     Poisson/Binomial negativa (función `glm_conteos`, documentada).
   - Si es continua y los supuestos se cumplen -> tabla F del ANOVA.
   - Si los supuestos fallan -> vía no paramétrica: Kruskal-Wallis por método
     + Scheirer-Ray-Hare (ANOVA tipo II sobre rangos) para la interacción.
3. **Sensibilidad LMM**: `variable ~ método + (1|aislamiento)` (modelo mixto
   con aislado aleatorio) para evaluar si la conclusión sobre el método es
   robusta; se reporta el ICC (proporción de varianza atribuible al aislado).
4. **Conidias**: diagnóstico de que `conidias_log10_ml` es continua (no
   entera), lo que **desactiva la rama Poisson/NB** (los conteos crudos no
   están disponibles y la escala ya es log10); se usa modelo lineal sobre
   log10.

**Por qué**: la filosofía del proyecto es NO elegir un test solo por producir
significancia; la elección se justifica con diagnóstico de datos.

**Supuestos**: OLS requiere residuos normales, homocedásticos e independientes;
el LMM requiere normalidad de los efectos aleatorios (aproximada).

**Cómo interpretar**:
- Reportar **F, p, eta², omega² e IC**; un p significativo con eta² pequeño
  no es biológicamente relevante.
- Un ICC alto indica que el aislado explica gran parte de la variación.
- La comparación entre el factorial y el LMM permite evaluar robustez.


In [9]:

# 7.1 Rendimiento de extracción
an_rendimiento = modelos.analisis_rendimiento(df_rend)
resultados["rendimiento"] = an_rendimiento
display(an_rendimiento["tabla_anova"])

# 7.2 Factorial por variable de respuesta
modelos_res = {}
lmm_res = {}
for variable in VARIABLES_RESPUESTA:
    m = modelos.analisis_factorial(df_bio, variable)
    modelos_res[variable] = m
    display(m["tabla_anova"])

# 7.3 Sensibilidad con modelo mixto (aislado aleatorio)
for variable in VARIABLES_RESPUESTA:
    lmm_res[variable] = modelos.analisis_sensibilidad_lmm(df_bio, variable)
    display(lmm_res[variable]["tabla_efectos_fijos"])

# 7.4 Diagnóstico de conidias
conidias_info = modelos.analisis_conidias(df_bio)
display(conidias_info["diagnostico"])

resultados["modelos"] = modelos_res
resultados["lmm"] = lmm_res
resultados["conidias"] = conidias_info


FASE 7 - Rendimiento de extracción (ANOVA de una vía)
Medias de rendimiento (%):
metodo_extraccion
maceracion     12.068
soxhlet        43.380
ultrasonido    17.053

Tabla ANOVA:
           fuente      sum_sq  df         F   PR(>F)  eta2_parcial  omega2_parcial
metodo_extraccion 1698.372614 2.0 50.841193 0.000173        0.9443          0.9172
         Residual  100.216331 6.0       NaN      NaN           NaN             NaN
eta2=0.9443 | omega2=0.9172 | Kruskal-Wallis p=0.0273


,fuente,sum_sq,df,F,PR(>F),eta2_parcial,omega2_parcial
0,metodo_extraccion,1698.372614,2.0,50.841193,0.000173,0.9443,0.9172
1,Residual,100.216331,6.0,NaN,NaN,NaN,NaN


FASE 7 - Análisis factorial: Crecimiento micelial (mm)
                       fuente       sum_sq    df          F       PR(>F)  eta2_parcial  omega2_parcial
            metodo_extraccion 16593.426523   2.0 244.072438 9.787070e-53        0.7241          0.7201
                  aislamiento  5256.788530  30.0   5.154801 8.478671e-13        0.4540          0.3648
metodo_extraccion:aislamiento  4233.017921  60.0   2.075446 1.076485e-04        0.4010          0.2071
                     Residual  6322.666667 186.0        NaN          NaN           NaN             NaN
Tipo de modelo: factorial_no_parametrico
Justificacion: Los supuestos del ANOVA factorial no se cumplen (Shapiro-Wilk p=0.0328; Levene p=0.0628). Se utiliza la via no parametrica (Kruskal-Wallis por metodo y Scheirer-Ray-Hare para la interaccion) como inferencia principal, conservando la tabla ANOVA y los tamanos de efecto como referencia descriptiva. Para %INH micelial, el efecto techo (muchos valores = 100) explica la violac

,fuente,sum_sq,df,F,PR(>F),eta2_parcial,omega2_parcial
0,metodo_extraccion,16593.426523,2.0,244.072438,9.787070e-53,0.7241,0.7201
1,aislamiento,5256.788530,30.0,5.154801,8.478671e-13,0.4540,0.3648
2,metodo_extraccion:aislamiento,4233.017921,60.0,2.075446,1.076485e-04,0.4010,0.2071
3,Residual,6322.666667,186.0,NaN,NaN,NaN,NaN


FASE 7 - Análisis factorial: Inhibición micelial (%)
                       fuente       sum_sq    df          F       PR(>F)  eta2_parcial  omega2_parcial
            metodo_extraccion 53840.335939   2.0 254.603435 5.598822e-54        0.7325          0.7285
                  aislamiento 21504.066747  30.0   6.779315 2.284802e-17        0.5223          0.4441
metodo_extraccion:aislamiento 13887.098547  60.0   2.189005 3.436219e-05        0.4139          0.2241
                     Residual 19666.471641 186.0        NaN          NaN           NaN             NaN
Tipo de modelo: factorial_ols
Justificacion: Los supuestos de normalidad y homocedasticidad se cumplen (Shapiro-Wilk y Levene p>0.05); el ANOVA factorial es la via adecuada. Nota: las replicas de %INH comparten el control C4 (pseudorreplicacion), lo que puede inflar levemente la precision.


...

FASE 7 - LMM sensibilidad: Inhibición de conidias (%)
                             efecto  coeficiente  error_estandar        t  p_valor  ic95_inferior  ic95_superior
                          Intercept      29.2394          1.7463  16.7432   0.0000        25.8166        32.6622
    C(metodo_extraccion)[T.soxhlet]     -24.8071          2.1790 -11.3848   0.0000       -29.0779       -20.5363
C(metodo_extraccion)[T.ultrasonido]     -29.4089          2.1790 -13.4968   0.0000       -33.6797       -25.1382
                          Group Var       0.0949          0.0563   1.6843   0.0921        -0.0155         0.2053
ICC (aislado)=0.0867; p_metodo (LRT)=0.0000


/home/mniev/.local/lib/python3.14/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


,efecto,coeficiente,error_estandar,t,p_valor,ic95_inferior,ic95_superior
0,Intercept,29.2394,1.7463,16.7432,0.0000,25.8166,32.6622
1,C(metodo_extraccion)[T.soxhlet],-24.8071,2.1790,-11.3848,0.0000,-29.0779,-20.5363
2,C(metodo_extraccion)[T.ultrasonido],-29.4089,2.1790,-13.4968,0.0000,-33.6797,-25.1382
3,Group Var,0.0949,0.0563,1.6843,0.0921,-0.0155,0.2053


FASE 7 - Análisis factorial: Conidias (log10/mL)
                       fuente     sum_sq    df          F       PR(>F)  eta2_parcial  omega2_parcial
            metodo_extraccion 238.944999   2.0 263.575621 5.233727e-55        0.7392          0.7354
                  aislamiento  98.931587  30.0   7.275302 1.098915e-18        0.5399          0.4645
metodo_extraccion:aislamiento 160.537024  60.0   5.902843 4.829518e-21        0.6557          0.5436
                     Residual  84.309333 186.0        NaN          NaN           NaN             NaN
Tipo de modelo: factorial_no_parametrico
Justificacion: Los supuestos del ANOVA factorial no se cumplen (Shapiro-Wilk p=0.0000; Levene p=0.0000). Se utiliza la via no parametrica (Kruskal-Wallis por metodo y Scheirer-Ray-Hare para la interaccion) como inferencia principal, conservando la tabla ANOVA y los tamanos de efecto como referencia descriptiva. Para %INH micelial, el efecto techo (muchos valores = 100) explica la violacion de normalida

,metrica,valor
0,n,279.0000
1,media,6.5624
2,varianza,2.0961
3,minimo,0.0000
4,maximo,8.6500
5,pct_valores_enteros,6.0900


## Fase 8: Comparaciones múltiples

**Qué se hace**: se comparan los métodos por pares según el modelo seleccionado
en la fase 7:
- Si el modelo fue paramétrico (ANOVA) -> **Tukey HSD** entre métodos.
- Si fue no paramétrico -> **test de Dunn** (manual, sobre rangos) con
  corrección **FDR** (Benjamini-Hochberg) y **Wilcoxon/Mann-Whitney** con FDR
  como robustez.
Se generan **letras compactas (CLD)**: métodos que comparten al menos una letra
NO difieren significativamente (p ajustada >= 0.05). Se guardan tablas y la
figura `posthoc_<variable>_letras`.

**Por qué**: comparar todas las técnicas por pares sin corregir infla el error
tipo I; las letras CLD facilitan la lectura de los grupos homogéneos.

**Supuestos**: el post-hoc debe corresponder al modelo principal (regla del
proyecto). Cuando la interacción método × aislado es significativa, la
comparación marginal entre métodos debe interpretarse con cautela: describe el
efecto promedio sobre los aislados.

**Cómo interpretar**: en cada variable, los métodos que comparten letra son
estadísticamente equivalentes al nivel 0.05 (ajustado); el orden de las medias
aporta la dirección del efecto.


In [10]:

posthoc_result = {}
for variable in VARIABLES_RESPUESTA:
    tipo_modelo = modelos_res[variable]["tipo_modelo"]
    posthoc_result[variable] = comparaciones.comparaciones_posthoc(df_bio, variable, tipo_modelo)
    display(posthoc_result[variable]["tabla_pares"])
resultados["posthoc"] = posthoc_result


FASE 8 - Post-hoc (Dunn (FDR)): Crecimiento micelial (mm)
                      par  estadistico_z  p_valor  p_valor_ajustado          letras
    maceracion vs soxhlet      -8.830402 0.000000          0.000000 sin letra comun
maceracion vs ultrasonido     -10.831393 0.000000          0.000000 sin letra comun
   soxhlet vs ultrasonido      -2.000991 0.045393          0.045393 sin letra comun
Letras CLD: {'ultrasonido': 'a', 'soxhlet': 'b', 'maceracion': 'c'}


,par,estadistico_z,p_valor,p_valor_ajustado,letras
0,maceracion vs soxhlet,-8.830402,0.000000,0.000000,sin letra comun
1,maceracion vs ultrasonido,-10.831393,0.000000,0.000000,sin letra comun
2,soxhlet vs ultrasonido,-2.000991,0.045393,0.045393,sin letra comun


FASE 8 - Post-hoc (Tukey HSD): Inhibición micelial (%)
                      par  diferencia_medias  p_valor_ajustado  ic95_inferior  ic95_superior  significativo          letras
    maceracion vs soxhlet         -26.756388          0.000000     -31.637087     -21.875688           True sin letra comun
maceracion vs ultrasonido         -31.584355          0.000000     -36.465054     -26.703656           True sin letra comun
   soxhlet vs ultrasonido          -4.827967          0.053256      -9.708667       0.052732          False               b
Letras CLD: {'maceracion': 'a', 'soxhlet': 'b', 'ultrasonido': 'b'}


,par,diferencia_medias,p_valor_ajustado,ic95_inferior,ic95_superior,significativo,letras
0,maceracion vs soxhlet,-26.756388,0.000000,-31.637087,-21.875688,True,sin letra comun
1,maceracion vs ultrasonido,-31.584355,0.000000,-36.465054,-26.703656,True,sin letra comun
2,soxhlet vs ultrasonido,-4.827967,0.053256,-9.708667,0.052732,False,b


FASE 8 - Post-hoc (Dunn (FDR)): Conidias (log10/mL)
                      par  estadistico_z  p_valor  p_valor_ajustado          letras
    maceracion vs soxhlet      -8.795389 0.000000          0.000000 sin letra comun
maceracion vs ultrasonido     -11.840785 0.000000          0.000000 sin letra comun
   soxhlet vs ultrasonido      -3.045396 0.002324          0.002324 sin letra comun
Letras CLD: {'ultrasonido': 'a', 'soxhlet': 'b', 'maceracion': 'c'}


,par,estadistico_z,p_valor,p_valor_ajustado,letras
0,maceracion vs soxhlet,-8.795389,0.000000,0.000000,sin letra comun
1,maceracion vs ultrasonido,-11.840785,0.000000,0.000000,sin letra comun
2,soxhlet vs ultrasonido,-3.045396,0.002324,0.002324,sin letra comun


FASE 8 - Post-hoc (Dunn (FDR)): Inhibición de conidias (%)
                      par  estadistico_z  p_valor  p_valor_ajustado          letras
    maceracion vs soxhlet       9.021910 0.000000          0.000000 sin letra comun
maceracion vs ultrasonido      11.835947 0.000000          0.000000 sin letra comun
   soxhlet vs ultrasonido       2.814036 0.004892          0.004892 sin letra comun
Letras CLD: {'maceracion': 'a', 'soxhlet': 'b', 'ultrasonido': 'c'}


,par,estadistico_z,p_valor,p_valor_ajustado,letras
0,maceracion vs soxhlet,9.021910,0.000000,0.000000,sin letra comun
1,maceracion vs ultrasonido,11.835947,0.000000,0.000000,sin letra comun
2,soxhlet vs ultrasonido,2.814036,0.004892,0.004892,sin letra comun


## Fase 9: Visualización de resultados

**Qué se hace**: se generan figuras de resultados (prefijo `resultados_`) para
cada variable: medias por método con SD / SE / IC95%, gráfico de interacción
método × aislado (medias por celda), efectos principales (método y aislado) y
comparación con letras CLD. Se guarda la tabla `medias_<variable>.csv`.

**Por qué**: las figuras de calidad de publicación permiten comunicar los
patrones de forma directa y verificable.

**Cómo interpretar**:
- Barras separadas en la figura CLD = grupos distintos.
- Un gráfico de interacción con líneas cruzadas sugiere que el efecto del
  método depende del aislado (respalda el término de interacción del ANOVA).
- Los IC95% que no se solapan indican diferencias descriptivas entre medias.


In [11]:

figuras_resultado = visualizar.figuras_resultados(df_bio, posthoc_result)
resultados["figuras_resultados"] = figuras_resultado


  Figuras de resultados generadas: 16


## Fase 10: Análisis multivariado de susceptibilidad

**Qué se hace**:
1. Se construye la **matriz por aislado** (31 x 6): media de %INH micelial y
   %INH de conidias para cada técnica; se estandariza (z-score).
2. **PCA**: varianza explicada, scree plot y biplot PC1-PC2.
3. **Clustering jerárquico** de Ward con dendrograma y coeficiente cofenético.
4. **KMeans** con codo (inercia) y silhouette; k óptimo automático.
5. **Categorías biológicas**: score compuesto de susceptibilidad (promedio de
   los z de inhibición micelial y de conidias) -> terciles -> etiquetas
   "Alta / Moderada / Baja susceptibilidad relativa". Se cruzan con los
   clusters de KMeans.
6. **Heatmap** de las 6 métricas estandarizadas con dendrograma y anotaciones
   de categoría; scatter de clusters en PC1-PC2.

**Por qué**: la susceptibilidad es un perfil multidimensional; PCA y clustering
la resumen objetivamente.

**Supuestos**: estandarización previa (las escalas de % difieren en
variabilidad); el número de clusters se elige por criterio objetivo
(silhouette).

**Cómo interpretar**:
- Mayor score compuesto = mayor susceptibilidad relativa (mayor inhibición).
- **No se usa el término "resistente"**: sin un umbral biológico validado, se
  habla de susceptibilidad relativa.
- Clusters con pocos aislados deben interpretarse con cautela (tamaño de
  muestra pequeño).


In [12]:

multivariado_info = multivariado.analisis_multivariado(df_bio)
resultados["multivariado"] = multivariado_info
display(multivariado_info["tabla_final"].head(15))
display(multivariado_info["cruce"])


FASE 10 - Análisis multivariado
Varianza explicada PC1-PC2: 42.2% / 29.2%
Coeficiente cofenetico (Ward): 0.7356
k optimo KMeans (silhouette): 2
Cruce cluster x categoria:
categoria_susceptibilidad  Alta susceptibilidad relativa  Baja susceptibilidad relativa  Moderada susceptibilidad relativa
cluster_kmeans                                                                                                            
0                                                      0                             10                                  0
1                                                     10                              1                                 10


,aislamiento,score_susceptibilidad,categoria_susceptibilidad,cluster_kmeans,inhib_micelial_maceracion,inhib_micelial_soxhlet,inhib_micelial_ultrasonido,inhib_conidias_maceracion,inhib_conidias_soxhlet,inhib_conidias_ultrasonido
11,H5N,0.923582,Alta susceptibilidad relativa,1,100.000000,62.222222,62.222222,28.442728,21.106821,11.025311
27,HC3,0.817429,Alta susceptibilidad relativa,1,100.000000,65.686275,60.294118,79.004329,-3.419913,9.220779
12,H6B,0.796077,Alta susceptibilidad relativa,1,92.820513,76.410256,63.076923,24.685139,12.510495,7.640638
15,H8N,0.656360,Alta susceptibilidad relativa,1,92.792793,74.774775,61.261261,26.770969,7.996523,7.431551
24,HC26,0.560286,Alta susceptibilidad relativa,1,100.000000,62.573099,62.573099,34.438550,11.759505,0.309461
16,H9N,0.522123,Alta susceptibilidad relativa,1,100.000000,67.619048,59.523810,25.759577,7.926024,4.359313
8,H4B,0.505390,Alta susceptibilidad relativa,1,93.137255,74.509804,60.784314,24.759515,8.531995,1.129235
1,FU2 (UCMU21),0.429572,Alta susceptibilidad relativa,1,100.000000,69.791667,43.229167,78.513189,2.206235,-2.494005
26,HC28,0.411930,Alta susceptibilidad relativa,1,100.000000,61.333333,57.333333,36.206897,16.003537,-5.923961
30,HC9,0.359129,Alta susceptibilidad relativa,1,100.000000,80.729167,60.416667,10.788382,0.368834,-0.645459


categoria_susceptibilidad,Alta susceptibilidad relativa,Baja susceptibilidad relativa,Moderada susceptibilidad relativa
cluster_kmeans,,,
0,0,10,0
1,10,1,10


## Fase 11: Ranking de técnicas de extracción

**Qué se hace**: por técnica se calculan tres métricas (rendimiento medio,
%INH micelial medio, %INH de conidias medio), se normalizan min-max a 0-1
(1 = mejor) y se promedian en un **score compuesto**; se ordenan las técnicas
y se genera un gráfico radar de tres ejes.

**Por qué**: integra rendimiento y actividad antifúngica en una única medida
de desempeño global, útil para la toma de decisiones.

**Supuestos**: las tres métricas se ponderan por igual (promedio simple); el
score es una herramienta descriptiva, no inferencial.

**Cómo interpretar**: la técnica con mayor score compuesto combina buen
rendimiento y alta actividad; verificar que las diferencias no contradigan las
comparaciones estadísticas de las fases 7-8 (una técnica puede liderar el
ranking pero no diferir significativamente de la segunda).


In [13]:

ranking_info = ranking.ranking_tecnicas(df_bio, df_rend)
resultados["ranking"] = ranking_info
display(ranking_info["tabla"])


FASE 11 - Ranking de técnicas de extracción
 ranking metodo_extraccion  rendimiento_medio_pct  inhib_micelial_medio_pct  inhib_conidias_medio_pct  rendimiento_norm  inhib_micelial_norm  inhib_conidias_norm  score_compuesto
       1        maceracion                 12.068                    86.349                    29.239             0.000                1.000                1.000            0.667
       2           soxhlet                 43.380                    59.593                     4.432             1.000                0.153                0.156            0.436
       3       ultrasonido                 17.053                    54.765                    -0.170             0.159                0.000                0.000            0.053


,ranking,metodo_extraccion,rendimiento_medio_pct,inhib_micelial_medio_pct,inhib_conidias_medio_pct,rendimiento_norm,inhib_micelial_norm,inhib_conidias_norm,score_compuesto
0,1,maceracion,12.068,86.349,29.239,0.000,1.000,1.000,0.667
1,2,soxhlet,43.380,59.593,4.432,1.000,0.153,0.156,0.436
2,3,ultrasonido,17.053,54.765,-0.170,0.159,0.000,0.000,0.053


## Fase 12: Informe final

**Qué se hace**: se genera `dca/resultados/reportes/informe_final.md` (español
profesional y neutro) con las secciones: resumen ejecutivo, calidad de datos,
diseño experimental, supuestos, análisis seleccionados y su justificación,
comparaciones múltiples, análisis multivariado, ranking, interpretación
biológica, conclusiones y limitaciones. Luego se convierte a
`informe_final.html` con un CSS simple y legible. Además se exporta un libro
Excel de resumen (`dca/resultados/excel/resumen_analisis.xlsx`) con las hojas
Descriptivos, Rendimiento (ANOVA), Factorial, Posthoc, Ranking y
Susceptibilidad.

**Por qué**: consolida todos los resultados en un documento accionable y
compartible, siguiendo la filosofía de reportar efecto, IC y p-valor.


In [14]:

informe.generar_informe(resultados)

# Libro Excel de resumen con las hojas solicitadas
from pipeline.config import exportar_excel, DIR_EXCEL

hojas_resumen = {
    "Descriptivos": tabla_desc,
    "Rendimiento_ANOVA": an_rendimiento["tabla_anova"],
    "Factorial_INH_micelial": modelos_res["porcentaje_inhibicion_micelial"]["tabla_anova"],
    "Posthoc_INH_micelial": posthoc_result["porcentaje_inhibicion_micelial"]["tabla_pares"],
    "Ranking": ranking_info["tabla"],
    "Susceptibilidad": multivariado_info["tabla_final"],
}
ruta_excel = exportar_excel(hojas_resumen, DIR_EXCEL / "resumen_analisis.xlsx")
print("Libro Excel de resumen:", ruta_excel)


FASE 12 - Informe final generado
  Markdown: /home/mniev/projects/proyecto_tomillo/dca/resultados/reportes/informe_final.md
  HTML:     /home/mniev/projects/proyecto_tomillo/dca/resultados/reportes/informe_final.html


Libro Excel de resumen: /home/mniev/projects/proyecto_tomillo/dca/resultados/excel/resumen_analisis.xlsx


## Conclusiones generales

1. **Rendimiento**: la técnica afectó significativamente el rendimiento de
   extracción; los resultados se reportan con tamaño de efecto (eta², omega²).
2. **Actividad antifúngica**: a 5 mg/mL los extractos inhibieron el crecimiento
   micelial con un marcado efecto techo en maceración; las comparaciones entre
   técnicas y aislados se apoyaron en la vía seleccionada por los supuestos.
3. **Conidias**: la variable es continua en escala log10; se modeló con un
   modelo lineal sobre log10 (la rama Poisson/NB no aplica y quedó documentada).
4. **Susceptibilidad**: los aislados se clasificaron en Alta / Moderada / Baja
   susceptibilidad relativa mediante score compuesto y clustering.
5. **Ranking**: la tabla de la fase 11 resume el desempeño global de las
   técnicas.

## Cómo re-ejecutar con un archivo nuevo

La pipeline está diseñada para reutilizarse sin modificar la lógica:

1. **Reemplazar las fuentes** editando las rutas en `pipeline/config.py`
   (`EXCEL_TIDY` y `CSV_RENDIMIENTO`) o, mejor, pasando rutas alternativas a
   `cargar_datos()` (acepta paths parametrizables).
2. **Respetar el esquema de columnas** del consolidado (7 columnas con los
   mismos nombres) o ajustar `RENOMBRES_BIO` en `pipeline/limpiar.py`. El
   master tendrá 7 columnas de respuesta más las 2 de control C4 cuando el
   Excel crudo del laboratorio esté disponible (9 columnas en total).
3. **Volver a generar y ejecutar**: `python3 generar_notebook_pipeline.py` y
   ejecutar el notebook completo.
4. **Revisar la validación** de la fase 3: si las dimensiones o el balanceo
   cambian, la validación lanzará un error y habrá que decidir cómo proceder
   (el pipeline nunca imputa datos ni elimina atípicos automáticamente).
5. **Controles y validación**: si el Excel crudo
   (`datos_crudos/dca/datos-proyectos tomillo-fusarium.xlsx`) no está presente,
   los controles C4 se omiten y la validación del %INH se saltea con un aviso;
   la inferencia no se ve afectada.

Los resultados se escriben siempre en `dca/resultados/` (tablas, figuras,
reportes y Excel), sin tocar los datos fuente.


### Nota final sobre interpretación

- **Significancia estadística** (p < 0.05) no implica **relevancia biológica**;
  siempre se evalúan junto con el tamaño de efecto y los IC95%.
- Las limitaciones principales (control compartido, %INH de conidias en escala
  log10 y ausencia de dosis-respuesta) están documentadas en la sección 10 del
  informe final.
- Todos los archivos generados quedan en `dca/resultados/`; el notebook solo
  orquesta la ejecución.
